# Data Access, Point-in-Time & Feature Validation

Walks the Step 3 layer end to end and shows the properties it guarantees.

**The flow (brief section 35):**

```
DataRepository -> PointInTimeView -> FeatureEngineer -> FeatureRepository -> X, y
```

**Prerequisite:**

```powershell
uv run ari generate-data --profile dev --seed 42
```

No models are trained here — that begins at Step 4.

In [ ]:
from __future__ import annotations

import sys
import warnings
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)

from app.services.container import Container  # noqa: E402
from data.repositories.availability import TABLE_AVAILABILITY  # noqa: E402
from data.repositories.base import ResultTruncatedError  # noqa: E402
from data.repositories.sampling import build_panel_sample, sample_training_period  # noqa: E402

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 4.5)
pd.set_option("display.width", 180)

repo = Container().data_repository
print(f"dataset: {repo.dataset_version()}")
print(f"tables : {len(repo.list_tables())}")

## 1. Truncation is loud, not silent

The defect Step 3 fixed. Every query carries a `LIMIT`, so a large slice used to come
back quietly cut short — and a feature computed over a truncated panel is not obviously
wrong. The lags simply stop early and the frame looks fine.

In [ ]:
try:
    repo.get_sales(start_date=date(2023, 1, 1), end_date=date(2025, 12, 31))
    print("returned silently — the guard is not working")
except ResultTruncatedError as exc:
    print(f"raised as intended:\n  {exc}")

bounded = repo.get_sales(start_date=date(2023, 1, 1), end_date=date(2025, 12, 31), max_rows=500)
print(f"\nexplicit max_rows=500 -> {len(bounded)} rows, no exception (opted in)")

## 2. Availability classes

"Clamp everything to the as-of date" is the obvious approach and it is wrong. A planner
on 1 June genuinely knows the promotion calendar for 10–24 June, and knows next Diwali's
date. Clamping those deletes information the business has.

In [ ]:
classes = pd.DataFrame(
    [{"table": t, "availability": a.value} for t, a in sorted(TABLE_AVAILABILITY.items())]
)
display(classes)

AS_OF = date(2025, 6, 30)
view = repo.as_of(AS_OF)
print(f"\nview: {view!r}")
print(f"dataset_version (as-of qualified): {view.dataset_version()}")

In [ ]:
products = repo.get_products()["product_id"].head(3).tolist()

rows = []
for label, frame, column in [
    ("sales_daily (observed)", view.get_sales(product_ids=products), "date"),
    ("inventory (observed)", view.get_inventory(product_ids=products), "date"),
    ("competitor (observed)", view.get_competitor_prices(product_ids=products), "date"),
    ("calendar (known ahead)", view.get_calendar(), "date"),
    ("promotions (known ahead)", view.get_promotions(product_ids=products), "start_date"),
]:
    latest = pd.to_datetime(frame[column]).dt.date.max()
    rows.append({
        "source": label,
        "latest date": latest,
        "past as-of?": "yes" if latest > AS_OF else "no",
    })

display(pd.DataFrame(rows))
print(f"as-of = {AS_OF}")
print("\nObserved tables stop at the as-of date. Planned tables do not — correctly.")

### 2a. The subtle half: future actuals are masked

A future promotion's *schedule* is knowable. Its realised spend is not — that is a
function of the demand a model is trying to predict.

In [ ]:
promos = view.get_promotions()
future = promos[pd.to_datetime(promos["start_date"]).dt.date > AS_OF]
past = promos[pd.to_datetime(promos["end_date"]).dt.date <= AS_OF]

print(f"promotions scheduled after as-of : {len(future):,}")
print(f"  their promotion_spend is null  : {future['promotion_spend'].isna().all()}")
print(f"  their promotion_units is null  : {future['promotion_units'].isna().all()}")
print(f"\nhistorical promotions           : {len(past):,}")
print(f"  spend retained                 : {past['promotion_spend'].notna().any()}")

display(future[["promotion_id", "start_date", "end_date", "discount_percentage",
                "promotion_spend", "promotion_units"]].head())

## 3. Sampling without loading everything

Sample the *keys*, then filter — rather than loading 6.7M rows to keep 5,000. Note
`co_listed=True`: sampling products and stores independently produces pairs that were
never stocked together, and most of that cross product returns nothing.

In [ ]:
sample = build_panel_sample(repo, n_products=10, n_stores=8, days=400, seed=42)
print(sample.describe())

(train_start, train_end), (test_start, test_end) = sample_training_period(
    repo, train_days=400, test_days=28
)
print(f"\ntrain {train_start} -> {train_end}")
print(f"test  {test_start} -> {test_end}")
print(f"chronological, no overlap: {train_end < test_start}")
print("\nA random split would let a model see the future while predicting the past.")

## 4. Building features

`FeatureEngineer` takes a **view**, never a bare repository. That is the structural half
of leakage prevention: it has no method that would return future observed data.

In [ ]:
from features.engineering import FeatureEngineer, FeatureRequest  # noqa: E402

try:
    FeatureEngineer(repo)
except TypeError as exc:
    print(f"bare repository refused:\n  {exc}\n")

train_view = repo.as_of(train_end)
engineer = FeatureEngineer(train_view)

panel = engineer.build(FeatureRequest(
    start_date=train_start,
    end_date=train_end,
    product_ids=sample.product_ids,
    store_ids=sample.store_ids,
))
print(f"panel: {len(panel):,} rows x {len(panel.columns)} columns")
print(f"dates: {panel['date'].min().date()} -> {panel['date'].max().date()}")

In [ ]:
from features.contracts import FEATURE_GROUPS  # noqa: E402

coverage = []
for name, group in FEATURE_GROUPS.items():
    present = [f for f in group.names() if f in panel.columns]
    coverage.append({
        "group": name.value,
        "declared": len(group.names()),
        "present": len(present),
        "example": present[0] if present else "-",
    })
display(pd.DataFrame(coverage))

## 5. Verifying the shift discipline

The mistake this guards against: `df.groupby(k).rolling(7)` reads perfectly well and
silently includes the current row — so `rolling_7_units` would contain one seventh of the
number being predicted.

In [ ]:
counts = panel.groupby(["product_id", "store_id"]).size()
pid, sid = counts.idxmax()
series = panel[(panel["product_id"] == pid) & (panel["store_id"] == sid)].sort_values("date").reset_index(drop=True)

i = 40
row, prior = series.iloc[i], series.iloc[i - 7]
manual_window = series.iloc[i - 7 : i]["units"]
including_today = series.iloc[i - 6 : i + 1]["units"]

print(f"series {pid} / {sid}, row dated {row['date'].date()}\n")
print(f"  units today                       : {row['units']}")
print(f"  lag_7_units                       : {row['lag_7_units']}")
print(f"  actual units 7 days earlier       : {prior['units']}   <- must match\n")
print(f"  rolling_7_units                   : {row['rolling_7_units']:.4f}")
print(f"  manual mean of prior 7 (correct)  : {manual_window.mean():.4f}   <- must match")
print(f"  mean INCLUDING today (the bug)    : {including_today.mean():.4f}   <- must NOT match")

In [ ]:
window = series.iloc[20:70]
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(window["date"], window["units"], lw=1.2, color="#a0aec0", label="units (target)")
ax.plot(window["date"], window["lag_7_units"], lw=1.8, color="#2b6cb0", label="lag_7_units")
ax.plot(window["date"], window["rolling_28_units"], lw=2.2, color="#c53030", label="rolling_28_units")
ax.set_title("Features trail the target — never anticipate it")
ax.set_ylabel("units")
ax.legend()
plt.tight_layout()

## 6. The leakage equivalence test

The definitive check, reproduced here. Build features twice — once from the full dataset,
once from a dataset physically truncated at the as-of date. For rows on or before that
date, a correct pipeline cannot tell the two apart.

It needs no knowledge of *which* feature might leak, which is what makes it keep holding
as Steps 4–11 add more.

In [ ]:
from data.repositories.availability import Availability, availability_of  # noqa: E402
from data.repositories.local import LocalDataRepository  # noqa: E402

GOLD = ROOT / "data" / "local" / "gold"
CUT = train_end - timedelta(days=45)

class TruncatedRepository(LocalDataRepository):
    """Observed tables genuinely end at the cutoff; planned tables do not."""
    def _cut(self, table, frame, column="date"):
        if availability_of(table) is not Availability.OBSERVED or frame.empty:
            return frame
        return frame[pd.to_datetime(frame[column]).dt.date <= CUT]
    def get_sales(self, **kw):
        return self._cut("sales_daily", super().get_sales(**kw))
    def get_inventory(self, **kw):
        return self._cut("inventory", super().get_inventory(**kw))
    def get_competitor_prices(self, **kw):
        return self._cut("competitor_pricing", super().get_competitor_prices(**kw))

request = FeatureRequest(
    start_date=CUT - timedelta(days=60), end_date=CUT,
    product_ids=sample.product_ids[:5], store_ids=sample.store_ids[:4],
)

full = FeatureEngineer(LocalDataRepository(GOLD, max_result_rows=5_000_000).as_of(CUT)).build(request)
cut = FeatureEngineer(TruncatedRepository(GOLD, max_result_rows=5_000_000).as_of(CUT)).build(request)

keys = ["date", "product_id", "store_id"]
a = full.sort_values(keys).reset_index(drop=True)
b = cut.sort_values(keys).reset_index(drop=True)

divergent = []
for column in [c for c in a.columns if c in b.columns]:
    x, y = a[column], b[column]
    if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
        if not np.allclose(x.astype(float).fillna(-9e9), y.astype(float).fillna(-9e9), atol=1e-9):
            divergent.append(column)
    elif not x.astype(str).equals(y.astype(str)):
        divergent.append(column)

print(f"full-world rows      : {len(a):,}")
print(f"truncated-world rows : {len(b):,}")
print(f"columns compared     : {len([c for c in a.columns if c in b.columns])}")
print(f"\ndivergent columns    : {divergent if divergent else 'NONE — no feature reads past the cut'}")

## 7. The three forward-looking features

Legitimately read beyond their row date, each with a written justification. Adding a
fourth requires editing the allow-list in `features.yaml`, which the tests assert against.

In [ ]:
from features.contracts import forward_looking_features  # noqa: E402

for spec in forward_looking_features():
    print(f"{spec.name}")
    print(f"  {spec.description}")
    print(f"  why: {spec.forward_justification}\n")

## 8. Dataset builders and lineage

In [ ]:
from features.datasets import (  # noqa: E402
    create_cross_price_dataset, create_forecasting_dataset,
    create_price_elasticity_dataset, create_promo_optimization_dataset,
    create_promo_uplift_dataset,
)

common = dict(product_ids=sample.product_ids, store_ids=sample.store_ids)

fc = create_forecasting_dataset(train_view, train_start=train_start, train_end=train_end, **common)
el = create_price_elasticity_dataset(train_view, start_date=train_start, end_date=train_end, **common)
up = create_promo_uplift_dataset(train_view, start_date=train_start, end_date=train_end, **common)
cp = create_cross_price_dataset(train_view, start_date=train_start, end_date=train_end,
                                store_ids=sample.store_ids, max_pairs=40)
op = create_promo_optimization_dataset(train_view, start_date=train_start, end_date=train_end)

summary = pd.DataFrame([
    {"dataset": ds.metadata.feature_set_name, "rows": len(ds.X),
     "features": len(ds.X.columns), "target": ds.metadata.target_name or "-"}
    for ds in (fc, el, up, cp, op)
])
display(summary)

print(fc.describe())

In [ ]:
print("elasticity dataset excludes the rows that bias the estimate:\n")
if "promotion_flag" in el.X.columns:
    print(f"  promotional rows remaining : {int(el.X['promotion_flag'].astype(bool).sum())}")
if "stockout_flag" in el.X.columns:
    print(f"  stockout rows remaining    : {int(el.X['stockout_flag'].astype(bool).sum())}")
print(f"  target                     : {el.metadata.target_name} (log-log, so the")
print("                               coefficient on log_price IS the elasticity)")

print("\nuplift dataset period labels:")
if not up.X.empty:
    display(up.X["period"].value_counts().to_frame("rows"))

## Findings

1. **Truncation raises** rather than silently returning a cut-short frame.
2. **Availability is per-table.** Observed data stops at the as-of date; planned data does
   not — because a planner genuinely knows next month's promotion calendar.
3. **Future actuals are masked** even on tables that are otherwise visible ahead.
4. **Lags and rolling windows trail the target exactly**, verified against manual
   arithmetic rather than asserted.
5. **The equivalence test finds no divergence** between the full and truncated worlds —
   no feature reads past the cut.
6. **Three features look forward**, each justified in the catalogue and pinned by the
   allow-list.
7. **All five builders** return separated `X`/`y` with lineage recording the dataset
   version, feature version and as-of date.

### Next

Step 4 — the baseline sales model, the first consumer of this layer and the first thing
scored against Step 2's hidden ground truth.